# Lesson 18: Best Practices & Safe-Retry Reliability Sidecars

In this notebook, you will:
1. Build a `sqlite3`-backed **Reliability Sidecar** for `create_support_ticket`.
2. Inject a simulated network drop after commit and verify crash-safe reconciliation.
3. Test fail-closed behavior and input-hash mismatch guards.

In [ ]:
import hashlib
import json
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE sidecar_ops (caller TEXT, op_key TEXT, input_hash TEXT, state TEXT, ticket_id TEXT, PRIMARY KEY (caller, op_key))")
conn.execute("CREATE TABLE ticket_system (ticket_id TEXT PRIMARY KEY, op_key TEXT UNIQUE, title TEXT)")

def call_create_ticket(caller: str, op_key: str, title: str, simulate_timeout: bool = False):
    in_hash = hashlib.sha256(json.dumps({"title": title}, sort_keys=True).encode()).hexdigest()
    claim = conn.execute("SELECT input_hash, state, ticket_id FROM sidecar_ops WHERE caller=? AND op_key=?", (caller, op_key)).fetchone()
    if claim:
        if claim[0] != in_hash:
            raise ValueError("Contract error: operation_key reused with different input!")
        if claim[2]:
            return {"ticket_id": claim[2], "status": "verified"}
        existing = conn.execute("SELECT ticket_id FROM ticket_system WHERE op_key=?", (op_key,)).fetchone()
        if existing:
            conn.execute("UPDATE sidecar_ops SET state='verified', ticket_id=? WHERE caller=? AND op_key=?", (existing[0], caller, op_key))
            return {"ticket_id": existing[0], "status": "verified"}
    conn.execute("INSERT INTO sidecar_ops VALUES (?, ?, ?, 'claimed', NULL)", (caller, op_key, in_hash))
    tid = f"T-{conn.execute('SELECT COUNT(*) FROM ticket_system').fetchone()[0] + 1:04d}"
    conn.execute("INSERT INTO ticket_system VALUES (?, ?, ?)", (tid, op_key, title))
    if simulate_timeout:
        raise TimeoutError("Connection lost after downstream commit!")
    conn.execute("UPDATE sidecar_ops SET state='verified', ticket_id=? WHERE caller=? AND op_key=?", (tid, caller, op_key))
    return {"ticket_id": tid, "status": "verified"}

try:
    call_create_ticket("user-1", "op-001", "Login broken", simulate_timeout=True)
except TimeoutError as e:
    print("First attempt:", e)

print("Retry result:", call_create_ticket("user-1", "op-001", "Login broken"))
print("Total rows in ticket_system:", conn.execute("SELECT COUNT(*) FROM ticket_system").fetchone()[0])

### Challenge Exercise
Add a `fail_closed_lookup()` branch where the downstream ticket system returns `503 Service Unavailable` during reconciliation, and verify the sidecar returns `{"status": "outcome_unknown"}` instead of blindly creating a duplicate.